In [42]:
# V5-01 — setup, paths, load linking_v5 output

from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)

ROOT = Path("/Users/davekokel/Projects/carp_v2")
if not ROOT.exists():
    raise FileNotFoundError(f"Expected repo root at {ROOT}, but it does not exist.")

BASE    = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW     = BASE / "raw"
WORKING = BASE / "working"

AUTO = ROOT / "seed_kits" / "2025-11-15-121231-autoload"

for p in [RAW, WORKING, AUTO]:
    if not p.exists():
        raise FileNotFoundError(f"Expected directory at {p}, but it does not exist.")

STRUCT_PATH = WORKING / "output_from_linking_v5.csv"

print("ROOT:       ", ROOT)
print("STRUCT_PATH:", STRUCT_PATH)

if not STRUCT_PATH.exists():
    raise FileNotFoundError(f"Structural CSV not found at {STRUCT_PATH}")

df_struct = pd.read_csv(STRUCT_PATH)

print("\nV5-01 — df_struct shape:", df_struct.shape)
print("V5-01 — df_struct columns:")
print(list(df_struct.columns))

print("\nV5-01 — basic ROI QC:")
print("  total rows:       ", len(df_struct))
print("  unique roi_dir:   ", df_struct["roi_dir"].nunique())

dups = df_struct[df_struct.duplicated("roi_dir", keep=False)].copy()
print("  duplicated roi_dir rows:", len(dups))

if not dups.empty:
    nunq = (
        dups.groupby("roi_dir")
        .nunique(dropna=False)
    )
    varying_cols = [
        c for c in nunq.columns
        if c != "roi_dir" and (nunq[c] > 1).any()
    ]
    print("\nV5-01 — columns with variation among duplicate roi_dir groups:", varying_cols)

    # Show a small sample of the problematic groups for future debugging
    sample_keys = nunq.index[(nunq[varying_cols] > 1).any(axis=1)].tolist()[:5]
    if sample_keys:
        print("\nV5-01 — SAMPLE duplicate roi_dir groups (for inspection only):")
        print(
            dups[dups["roi_dir"].isin(sample_keys)]
            .sort_values(["roi_dir"])
            .head(40)
        )

    # For v5 we accept that duplicates exist and collapse to 1 row per ROI.
    # We keep the first row per roi_dir; upstream linking_v5 should already
    # have consistent biological fields, and differences here are treated as
    # nuisance metadata.
    print("\nV5-01 — collapsing to 1 row per roi_dir (keep='first').")
    df_struct = (
        df_struct
        .sort_values(["roi_dir"])
        .drop_duplicates("roi_dir", keep="first")
        .reset_index(drop=True)
    )

print("\nV5-01 — df_struct AFTER potential collapse:")
print("  total rows:", len(df_struct))
print("  unique roi_dir:", df_struct["roi_dir"].nunique())

ROOT:        /Users/davekokel/Projects/carp_v2
STRUCT_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/output_from_linking_v5.csv

V5-01 — df_struct shape: (1082, 46)
V5-01 — df_struct columns:
['date_experiment', 'fish', 'roi_rel', 'roi_name', 'roi_tiffs', 'roi_dir', 'dataset', 'experiment_folder', 'roi_path_date_yyyymmdd', 'fish_folder', 'roi_folder', 'fish_id', 'fish_number', 'fish_age_hpf', 'fish_nickname', 'fish_raw_norm', 'roi_anatomy_tokens', 'roi_anatomy', 'fish_folder_patched', 'dataset_slug', 'dataset_slug_norm', 'sheet_slug_norm', 'date_mount_yyyymmdd', 'date_mount', 'Date imaged', 'mount_id', 'ZF female genotype', 'ZF male genotype', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'Imaged Locations', 'Unique Targets with blanks', 'Unique Targets', 'Data location', 'link_source', 'plate_date', 'mount_id_inferred', 'mount_id_source', 'plate_key', 'plate_id_fill

In [43]:
# V5-02 — load reference catalogs (parents, constructs, fluors, tags, injected mapping)

from pathlib import Path
import pandas as pd

# parent map can be either XLSX or CSV (v5)
PARENT_MAP_XLSX = RAW / "Unique_parent_names__mom_dad_combined__preview_dqm_v5.xlsx"
PARENT_MAP_CSV  = RAW / "Unique_parent_names__mom_dad_combined__preview_dqm_v5.csv"

INJECTED_PLASMID_PATH = RAW / "Unique_injected_plasmid__preview_dqm.xlsx"
INJECTED_RNA_PATH     = RAW / "Unique_injected_rna__preview_dqm.xlsx"
PLASMIDS_JANELIA_PATH = RAW / "plasmids_janelia_googlesheet.xlsx"

CONSTRUCTS_PATH = AUTO / "constructs_plasmid.csv"
FLUORS_PATH     = AUTO / "fluors.csv"
TAGS_PATH       = AUTO / "tags.xlsx"
ALIAS_PATH      = AUTO / "alias.csv"

print("PARENT_MAP_XLSX:   ", PARENT_MAP_XLSX)
print("PARENT_MAP_CSV:    ", PARENT_MAP_CSV)
print("INJECTED_PLASMID:  ", INJECTED_PLASMID_PATH)
print("INJECTED_RNA:      ", INJECTED_RNA_PATH)
print("PLASMIDS_JANELIA:  ", PLASMIDS_JANELIA_PATH)
print("CONSTRUCTS_PATH:   ", CONSTRUCTS_PATH)
print("FLUORS_PATH:       ", FLUORS_PATH)
print("TAGS_PATH:         ", TAGS_PATH)
print("ALIAS_PATH:        ", ALIAS_PATH)

# ── pick parent_map path (prefer XLSX if it exists, else CSV) ─────────────────
if PARENT_MAP_XLSX.exists():
    PARENT_MAP_PATH = PARENT_MAP_XLSX
    parent_map_loader = "excel"
elif PARENT_MAP_CSV.exists():
    PARENT_MAP_PATH = PARENT_MAP_CSV
    parent_map_loader = "csv"
else:
    raise FileNotFoundError(
        "Could not find parent map v5 in RAW; expected one of:\n"
        f"  {PARENT_MAP_XLSX}\n"
        f"  {PARENT_MAP_CSV}"
    )

# ── check other required paths ────────────────────────────────────────────────
missing = [p for p in [
    INJECTED_PLASMID_PATH,
    INJECTED_RNA_PATH,
    PLASMIDS_JANELIA_PATH,
    CONSTRUCTS_PATH,
    FLUORS_PATH,
    TAGS_PATH,
    ALIAS_PATH,
] if not p.exists()]

if missing:
    print("\nMISSING path(s):")
    for p in missing:
        print("  ", p)
    raise FileNotFoundError("Some required catalog files are missing (see list above).")

# ── load catalogs ─────────────────────────────────────────────────────────────
print("\nUsing parent_map file:", PARENT_MAP_PATH, f"(loader={parent_map_loader})")
if parent_map_loader == "excel":
    parent_map = pd.read_excel(PARENT_MAP_PATH)
else:
    parent_map = pd.read_csv(PARENT_MAP_PATH)

injected_plasmid = pd.read_excel(INJECTED_PLASMID_PATH)
injected_rna     = pd.read_excel(INJECTED_RNA_PATH)
plasmids_sheet   = pd.read_excel(PLASMIDS_JANELIA_PATH)

constructs       = pd.read_csv(CONSTRUCTS_PATH)
fluors           = pd.read_csv(FLUORS_PATH)
tags_cat         = pd.read_excel(TAGS_PATH)
alias            = pd.read_csv(ALIAS_PATH)

print("\nV5-02 — shapes:")
print("  parent_map:      ", parent_map.shape)
print("  injected_plasmid:", injected_plasmid.shape)
print("  injected_rna:    ", injected_rna.shape)
print("  plasmids_sheet:  ", plasmids_sheet.shape)
print("  constructs:      ", constructs.shape)
print("  fluors:          ", fluors.shape)
print("  tags_cat:        ", tags_cat.shape)
print("  alias:           ", alias.shape)

print("\nV5-02 — parent_map columns:", list(parent_map.columns))
print("V5-02 — constructs columns:", list(constructs.columns))
print("V5-02 — fluors columns:    ", list(fluors.columns))
print("V5-02 — tags_cat columns:  ", list(tags_cat.columns))
print("V5-02 — injected_plasmid columns:", list(injected_plasmid.columns))
print("V5-02 — injected_rna columns:    ", list(injected_rna.columns))

# normalize parent_map → expected columns:
# ['parent_fish_name', 'plasmid_base_code', 'allele', 'injected_rna', 'injected_plasmid']
parent_cols_norm = {
    "parent_fish_name":  "parent_fish_name",
    "parent_name":       "parent_fish_name",
    "plasmid_base_code": "plasmid_base_code",
    "allele":            "allele",
    "injected_rna":      "injected_rna",
    "injected plasmid":  "injected_plasmid",
    "injected_plasmid":  "injected_plasmid",
}
parent_map = parent_map.rename(
    columns={k: v for k, v in parent_cols_norm.items() if k in parent_map.columns}
)

needed_parent_cols = ["parent_fish_name", "plasmid_base_code", "allele", "injected_rna", "injected_plasmid"]
missing_parent_cols = [c for c in needed_parent_cols if c not in parent_map.columns]
if missing_parent_cols:
    raise KeyError(f"parent_map missing expected columns: {missing_parent_cols}")

print("\nV5-02 — parent_map normalized columns:", list(parent_map.columns))

PARENT_MAP_XLSX:    /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_parent_names__mom_dad_combined__preview_dqm_v5.xlsx
PARENT_MAP_CSV:     /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_parent_names__mom_dad_combined__preview_dqm_v5.csv
INJECTED_PLASMID:   /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_injected_plasmid__preview_dqm.xlsx
INJECTED_RNA:       /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_injected_rna__preview_dqm.xlsx
PLASMIDS_JANELIA:   /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/plasmids_janelia_googlesheet.xlsx
CONSTRUCTS_PATH:    /Users/davekokel/Projects/carp_v2/seed_kits/2025-11-15-121231-autoload/constructs_plasmid.csv
FLUORS_PATH:        /Users/davekokel/Projects/carp_v2/seed_kits/2025-11-15-121231-autoload/fluors.csv
TAGS_PATH:          /Users/davekokel/Projects/carp_v2/seed_kits/2025-11-15-121231-autoload/tags.xlsx
ALIAS_PATH: 

In [44]:
# V5-03 — parent/genotype enrichment from parent_map

df_enrich = df_struct.copy()
print("V5-03 — starting df_enrich shape:", df_enrich.shape)

# We expect these columns from linking_v5:
expected_parent_name_cols = ["ZF female genotype", "ZF male genotype"]
missing_parent_name_cols = [c for c in expected_parent_name_cols if c not in df_enrich.columns]
if missing_parent_name_cols:
    raise KeyError(f"df_struct is missing expected parent name columns: {missing_parent_name_cols}")

df_enrich = df_enrich.rename(
    columns={
        "ZF female genotype": "parent_female_name",
        "ZF male genotype": "parent_male_name",
    }
)

# strip whitespace and normalize case minimally
for col in ["parent_female_name", "parent_male_name"]:
    df_enrich[col] = df_enrich[col].astype(str).str.strip().replace({"nan": np.nan})

parent_map_small = parent_map[["parent_fish_name", "plasmid_base_code", "allele"]].copy()
parent_map_small["parent_fish_name_norm"] = parent_map_small["parent_fish_name"].astype(str).str.strip()

for side in ["female", "male"]:
    src_col = f"parent_{side}_name"
    norm_col = f"{src_col}_norm"
    df_enrich[norm_col] = df_enrich[src_col].astype(str).str.strip()

    df_enrich = df_enrich.merge(
        parent_map_small[["parent_fish_name_norm", "plasmid_base_code", "allele"]],
        left_on=norm_col,
        right_on="parent_fish_name_norm",
        how="left",
        suffixes=("", f"_{side}"),
    ).drop(columns=["parent_fish_name_norm"])

# female side columns: plasmid_base_code_female, allele_female
# male side columns:   plasmid_base_code_male,   allele_male

# build genotype_base_codes / genotype_allele_codes as pipe-delimited lists
geno_base = []
geno_alle = []

for _, row in df_enrich.iterrows():
    bases = []
    alleles = []
    for side in ["female", "male"]:
        bc = row.get(f"plasmid_base_code_{side}")
        al = row.get(f"allele_{side}")
        if pd.notna(bc):
            bases.append(str(bc).strip())
            if pd.notna(al):
                alleles.append(str(al).strip())
    if bases:
        geno_base.append("|".join(bases))
        geno_alle.append("|".join(alleles) if alleles else None)
    else:
        geno_base.append(None)
        geno_alle.append(None)

df_enrich["genotype_base_codes"]   = geno_base
df_enrich["genotype_allele_codes"] = geno_alle

print("\nV5-03 — genotype coverage:")
print("  rows with genotype_base_codes:", df_enrich["genotype_base_codes"].notna().sum(), "/", len(df_enrich))

print("\nV5-03 — sample rows with genotype_base_codes:")
display(
    df_enrich.loc[df_enrich["genotype_base_codes"].notna(),
                  ["roi_dir", "parent_female_name", "parent_male_name", "genotype_base_codes", "genotype_allele_codes"]]
    .head(20)
)

print("\nV5-03 — sample rows with NO genotype_base_codes (for inspection):")
display(
    df_enrich.loc[df_enrich["genotype_base_codes"].isna(),
                  ["roi_dir", "parent_female_name", "parent_male_name"]]
    .head(20)
)

V5-03 — starting df_enrich shape: (976, 46)

V5-03 — genotype coverage:
  rows with genotype_base_codes: 754 / 976

V5-03 — sample rows with genotype_base_codes:


,roi_dir,parent_female_name,parent_male_name,genotype_base_codes,genotype_allele_codes
5,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
6,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
7,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
8,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
9,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
10,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
11,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
12,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
13,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane Halo,membrane Halo,pSWIN01,is01
14,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,membrane mChilada,membrane mChilada,pDQM082,315



V5-03 — sample rows with NO genotype_base_codes (for inspection):


,roi_dir,parent_female_name,parent_male_name
0,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN
1,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN
2,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN
3,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN
4,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN
27,/clusterfs/vast/abcabc/Aang_Foundation/2025080...,NaN,NaN
28,/clusterfs/vast/abcabc/Aang_Foundation/2025080...,NaN,NaN
29,/clusterfs/vast/abcabc/Aang_Foundation/2025080...,NaN,NaN
30,/clusterfs/vast/abcabc/Aang_Foundation/2025080...,NaN,NaN
31,/clusterfs/vast/abcabc/Aang_Foundation/2025080...,NaN,NaN


In [45]:
# V5-04 — build genotype marker rollup from constructs + fluors + tags (cytosol default)

# Normalize constructs columns
constructs_norm = constructs.rename(
    columns={
        "code": "plasmid_code",
        "plasmid_code": "plasmid_code",
    }
)
if "fluor_code" not in constructs_norm.columns or "tag_code" not in constructs_norm.columns:
    raise KeyError("constructs_plasmid.csv must contain at least 'fluor_code' and 'tag_code' columns.")

# Normalize fluors → use nickname as fluor_code
fluors_norm = fluors.rename(columns={"nickname": "fluor_code"})
fluors_norm["fluor_code"] = fluors_norm["fluor_code"].astype(str).str.strip()

# Normalize tags → nickname as tag_code, localization as tag_localization
tags_norm = tags_cat.rename(columns={"nickname": "tag_code", "localization": "tag_localization"})
tags_norm["tag_code"] = tags_norm["tag_code"].astype(str).str.strip()

# Cytosol default tag for untagged fluors:
# If constructs row has fluor_code but tag_code is NaN, synthesize tag_code="cytosol"
# and tag_localization="cytosol".
constructs_ft = constructs_norm.copy()
constructs_ft["fluor_code"] = constructs_ft["fluor_code"].astype(str).str.strip()
constructs_ft["tag_code"]   = constructs_ft["tag_code"].astype(str).str.strip()

mask_no_tag = constructs_ft["fluor_code"].notna() & (constructs_ft["fluor_code"] != "") & (
    constructs_ft["tag_code"].isna() | (constructs_ft["tag_code"] == "") | (constructs_ft["tag_code"].str.lower() == "nan")
)

constructs_ft.loc[mask_no_tag, "tag_code"] = "cytosol"

# Add cytosol tag row to tags_norm if missing
if "cytosol" not in set(tags_norm["tag_code"].astype(str).str.strip()):
    tags_norm = pd.concat(
        [
            tags_norm,
            pd.DataFrame(
                [{
                    "tag_code": "cytosol",
                    "tag_localization": "cytosol",
                    "note": "default tag for untagged fluor (assumed cytosolic)",
                    "citation_link": np.nan,
                    "aliases": np.nan,
                }]
            ),
        ],
        ignore_index=True,
    )

# Join constructs → tags to get tag_localization
constructs_ft = constructs_ft.merge(
    tags_norm[["tag_code", "tag_localization"]],
    on="tag_code",
    how="left",
)

# Per-plasmid marker info
constructs_ft_small = (
    constructs_ft[["plasmid_code", "fluor_code", "tag_code", "tag_localization"]]
    .drop_duplicates()
)

print("\nV5-04 — constructs_ft_small sample:")
display(constructs_ft_small.head(20))

# Explode genotype_base_codes to one row per (roi_dir, plasmid_code)
geno = df_enrich[["roi_dir", "genotype_base_codes"]].copy()

def split_codes(val):
    if pd.isna(val) or not isinstance(val, str) or not val.strip():
        return []
    return [x.strip() for x in val.split("|") if x.strip()]

geno = geno.assign(plasmid_code_list=geno["genotype_base_codes"].apply(split_codes))
geno = geno.explode("plasmid_code_list")
geno = geno.rename(columns={"plasmid_code_list": "plasmid_code"})
geno = geno[geno["plasmid_code"].notna() & (geno["plasmid_code"] != "")].copy()

print("\nV5-04 — geno (exploded genotype_base_codes) sample:")
display(geno.head(20))

# Join with constructs_ft_small → per-ROI marker rows
geno_markers = geno.merge(
    constructs_ft_small,
    on="plasmid_code",
    how="left",
)

print("\nV5-04 — geno_markers sample:")
display(geno_markers.head(20))

# Aggregate per ROI into pipe-delimited marker lists
def agg_marker_lists(group: pd.DataFrame) -> pd.Series:
    def uniq_pipe(col):
        vals = [str(x).strip() for x in group[col] if pd.notna(x) and str(x).strip()]
        if not vals:
            return np.nan
        seen = []
        for v in vals:
            if v not in seen:
                seen.append(v)
        return "|".join(seen)

    fluor_codes = uniq_pipe("fluor_code")
    tag_codes   = uniq_pipe("tag_code")
    locs        = uniq_pipe("tag_localization")

    # fusion labels like fluor(tag_localization)
    fusions = []
    if pd.notna(fluor_codes) and pd.notna(locs):
        fluor_list = fluor_codes.split("|")
        loc_list   = locs.split("|")
        for f in fluor_list:
            for loc in loc_list:
                fusions.append(f"{f}({loc})")
    fusion_labels = "|".join(sorted(set(fusions))) if fusions else np.nan

    return pd.Series(
        {
            "genotype_marker_fluor_codes": fluor_codes,
            "genotype_marker_tag_codes": tag_codes,
            "genotype_marker_localizations": locs,
            "genotype_marker_fusion_labels": fusion_labels,
        }
    )

geno_agg = (
    geno_markers
    .groupby("roi_dir", as_index=False)
    .apply(agg_marker_lists)
)

print("\nV5-04 — geno_agg (per-ROI marker rollup) sample:")
display(geno_agg.head(20))

# Merge geno_agg back into df_enrich
df_enrich = df_enrich.merge(  
    geno_agg,
    on="roi_dir",
    how="left",
)

print("\nV5-04 — df_enrich after marker rollup merge:")
print("  shape:", df_enrich.shape)
print("  new marker columns:",
      [c for c in ["genotype_marker_fluor_codes",
                   "genotype_marker_tag_codes",
                   "genotype_marker_localizations",
                   "genotype_marker_fusion_labels"]
       if c in df_enrich.columns])

print("\nV5-04 — sample enriched rows (with markers):")
display(
    df_enrich.loc[
        df_enrich["genotype_marker_fluor_codes"].notna(),
        [
            "roi_dir",
            "genotype_base_codes",
            "genotype_marker_fluor_codes",
            "genotype_marker_tag_codes",
            "genotype_marker_localizations",
            "genotype_marker_fusion_labels",
        ],
    ].head(20)
)


V5-04 — constructs_ft_small sample:


,plasmid_code,fluor_code,tag_code,tag_localization
0,pDQM001,mSG,cytosol,cytosol
1,pDQM002,mSG,cytosol,cytosol
2,pDQM005,tdmSG,2xLynk,membrane
3,pDQM006,tdmSG,cytosol,cytosol
4,pDQM007,tdmSG,cytosol,cytosol
5,pDQM008,tdmSG,cytosol,cytosol
6,pDQM009,mScarlet,cytosol,cytosol
7,pDQM009,mKate2,cytosol,cytosol
8,pDQM009,Electra2,cytosol,cytosol
9,pDQM009,mKOK,cytosol,cytosol



V5-04 — geno (exploded genotype_base_codes) sample:


,roi_dir,genotype_base_codes,plasmid_code
5,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
6,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
7,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
8,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
9,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
10,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
11,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
12,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
13,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pSWIN01,pSWIN01
14,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,pDQM082



V5-04 — geno_markers sample:


...


V5-04 — geno_agg (per-ROI marker rollup) sample:


/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_83558/1974207367.py:132: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(agg_marker_lists)


,roi_dir,genotype_marker_fluor_codes,genotype_marker_tag_codes,genotype_marker_localizations,genotype_marker_fusion_labels
0,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
1,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
2,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
3,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
4,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
5,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
6,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
7,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
8,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
9,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,2xLynk,membrane,tdmChilada(membrane)



V5-04 — df_enrich after marker rollup merge:
  shape: (976, 58)
  new marker columns: ['genotype_marker_fluor_codes', 'genotype_marker_tag_codes', 'genotype_marker_localizations', 'genotype_marker_fusion_labels']

V5-04 — sample enriched rows (with markers):


,roi_dir,genotype_base_codes,genotype_marker_fluor_codes,genotype_marker_tag_codes,genotype_marker_localizations,genotype_marker_fusion_labels
14,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
15,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
16,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
17,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
18,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
19,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
20,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
21,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
22,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)
23,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,pDQM082,tdmChilada,2xLynk,membrane,tdmChilada(membrane)


In [46]:
# V5-10 — dataset-level parent map patch (skittles, mem-mito, mem-histone, etc.)

import re

if "df_enrich" not in globals():
    raise NameError("V5-10: df_enrich not found; run V5-03/V5-04 first.")
if "parent_map" not in globals():
    raise NameError("V5-10: parent_map not found; run V5-02 first.")

# ─────────────────────────────────────────────
# 1) Normalize experiment_folder → exp_slug_norm
#    and parent_fish_name → parent_slug_norm
# ─────────────────────────────────────────────

def _norm_label(s: str | float | None) -> str | None:
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    # strip leading YYYYMMDD_ or YYYYMMDD-
    s = re.sub(r"^\d{8}[_-]", "", s)
    # drop trailing parenthetical pieces like "(fish1-3)"
    s = re.sub(r"\(.*?\)", "", s)
    # collapse to alphanumeric only (mem-mito, mem_mito, mem mito → memmito)
    s = re.sub(r"[^a-z0-9]+", "", s)
    s = s.strip()
    return s or None

df_enrich = df_enrich.copy()

df_enrich["exp_slug_norm"] = df_enrich["experiment_folder"].apply(_norm_label)
parent_map = parent_map.copy()
parent_map["parent_slug_norm"] = parent_map["parent_fish_name"].apply(_norm_label)

print("V5-10 — slug coverage:")
print("  df_enrich unique exp_slug_norm:", df_enrich["exp_slug_norm"].nunique())
print("  parent_map unique parent_slug_norm:", parent_map["parent_slug_norm"].nunique())

# ─────────────────────────────────────────────
# 2) Aggregate parent_map by slug → geno + injection names
# ─────────────────────────────────────────────

def _agg_nonempty(series: pd.Series) -> str | None:
    vals = [
        str(x).strip()
        for x in series
        if pd.notna(x) and str(x).strip() not in ("", "nan", "n/a", "na")
    ]
    if not vals:
        return None
    # keep order, drop duplicates
    seen = set()
    out = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    # join with '|' so downstream code treats multi as expected
    return "|".join(out)

parent_slug_agg = (
    parent_map
    .groupby("parent_slug_norm", dropna=True)
    .agg({
        "plasmid_base_code": _agg_nonempty,
        "allele": _agg_nonempty,
        "injected_rna": _agg_nonempty,
        "injected_plasmid": _agg_nonempty,
    })
    .reset_index()
    .rename(columns={
        "plasmid_base_code": "geno_base_codes_v5",
        "allele": "geno_alleles_v5",
        "injected_rna": "inj_rna_names_v5",
        "injected_plasmid": "inj_plasmid_names_v5",
    })
)

print("\nV5-10 — parent_slug_agg sample:")
print(parent_slug_agg.head(20))

# ─────────────────────────────────────────────
# 3) Join slug-agg onto df_enrich
# ─────────────────────────────────────────────

df_enrich = df_enrich.merge(
    parent_slug_agg,
    how="left",
    left_on="exp_slug_norm",
    right_on="parent_slug_norm",
)

print("\nV5-10 — df_enrich after slug join:")
print("  shape:", df_enrich.shape)
print("  columns added:",
      [c for c in ["parent_slug_norm", "geno_base_codes_v5", "geno_alleles_v5",
                   "inj_rna_names_v5", "inj_plasmid_names_v5"]
       if c in df_enrich.columns])

# ─────────────────────────────────────────────
# 4) Patch genotype_base_codes / genotype_allele_codes
#    ONLY where they are currently empty AND we have v5 values
# ─────────────────────────────────────────────

def _is_empty_str_col(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip().isin(["", "None", "nan"]))

mask_geno_missing = _is_empty_str_col(df_enrich["genotype_base_codes"])
mask_allele_missing = _is_empty_str_col(df_enrich["genotype_allele_codes"])

mask_geno_has_v5 = df_enrich["geno_base_codes_v5"].notna()
mask_allele_has_v5 = df_enrich["geno_alleles_v5"].notna()

to_patch_geno = mask_geno_missing & mask_geno_has_v5
to_patch_allele = mask_allele_missing & mask_allele_has_v5

print("\nV5-10 — genotype patch counts:")
print("  rows with empty genotype_base_codes:", int(mask_geno_missing.sum()))
print("  rows with v5 geno_base_codes_v5:", int(mask_geno_has_v5.sum()))
print("  rows patched for genotype_base_codes:", int(to_patch_geno.sum()))
print("  rows patched for genotype_allele_codes:", int(to_patch_allele.sum()))

df_enrich.loc[to_patch_geno, "genotype_base_codes"] = df_enrich.loc[to_patch_geno, "geno_base_codes_v5"]
df_enrich.loc[to_patch_allele, "genotype_allele_codes"] = df_enrich.loc[to_patch_allele, "geno_alleles_v5"]

# ─────────────────────────────────────────────
# 5) Patch treatment name columns (sheet names)
#    so later V5-05 mapping can turn them into base codes
# ─────────────────────────────────────────────

for col_name, agg_name in [
    ("treatment_rna_names_sheet", "inj_rna_names_v5"),
    ("treatment_plasmid_names_sheet", "inj_plasmid_names_v5"),
]:
    if col_name not in df_enrich.columns:
        continue
    if agg_name not in df_enrich.columns:
        continue

    mask_name_missing = _is_empty_str_col(df_enrich[col_name])
    mask_name_has_v5 = df_enrich[agg_name].notna()

    to_patch_name = mask_name_missing & mask_name_has_v5

    print(f"\nV5-10 — patching {col_name} from {agg_name}:")
    print("  rows with empty", col_name, ":", int(mask_name_missing.sum()))
    print("  rows with", agg_name, ":", int(mask_name_has_v5.sum()))
    print("  rows patched for", col_name, ":", int(to_patch_name.sum()))

    df_enrich.loc[to_patch_name, col_name] = df_enrich.loc[to_patch_name, agg_name]

# ─────────────────────────────────────────────
# 6) Quick sanity check on key slug families (skittles, skittlez, mem-mito, mem-histone)
# ─────────────────────────────────────────────

def _slug_contains(df: pd.DataFrame, needle: str) -> pd.DataFrame:
    return df[df["exp_slug_norm"].fillna("").str.contains(needle, na=False)]

for needle in ["skittle", "memmito", "memhistone", "mitomsg"]:
    sub = _slug_contains(df_enrich, needle)
    if not sub.empty:
        print(f"\nV5-10 — sample rows for slug containing '{needle}':")
        print(
            sub[
                [
                    "roi_dir",
                    "experiment_folder",
                    "exp_slug_norm",
                    "genotype_base_codes",
                    "genotype_allele_codes",
                    "treatment_plasmid_names_sheet" if "treatment_plasmid_names_sheet" in sub.columns else "",
                    "treatment_rna_names_sheet" if "treatment_rna_names_sheet" in sub.columns else "",
                    "geno_base_codes_v5",
                    "geno_alleles_v5",
                    "inj_rna_names_v5",
                    "inj_plasmid_names_v5",
                ]
            ].head(10)
        )

print("\nV5-10 — done. Now re-run V5-05+ cells (treatment mapping, marker rollup, organelles, outputs).")

V5-10 — slug coverage:
  df_enrich unique exp_slug_norm: 32
  parent_map unique parent_slug_norm: 41

V5-10 — parent_slug_agg sample:
                               parent_slug_norm geno_base_codes_v5      geno_alleles_v5 inj_rna_names_v5 inj_plasmid_names_v5
0                                           abe            pDQM034                  309             None                 None
1                                           ben            pDQM034                  310             None                 None
2                                     casperrnf               None                 None             None                 None
3                                         chris            pDQM034                  317             None                 None
4                                  csppiglet14a               None                 None             None                 None
5                                        dennis            pDQM036                  318             None      

KeyError: "[''] not in index"

In [47]:
# V5-11 — explicit patch for remaining mem-mito / mem-histone holes

import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V5-11: df_enrich not found; run V5-03+ and V5-10 first.")

df_enrich = df_enrich.copy()

def _is_empty(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip().isin(["", "None", "nan", "NaN"]))

# ─────────────────────────────────────────────
# 1) Patch mem-mito datasets with no geno/treatment
#    Canonical mapping we agreed:
#      geno: pDQM082 (allele 315)
#      mRNA: MGCO-01  (mitochondria)
# ─────────────────────────────────────────────

mask_mem_mito_ds = df_enrich["dataset_slug"].astype(str).str.contains("mem-mito", case=False, na=False)
mask_geno_empty  = _is_empty(df_enrich["genotype_base_codes"])
mask_treat_rna_empty = _is_empty(df_enrich.get("treatment_rna_names_sheet", pd.Series([None]*len(df_enrich))))

mem_mito_to_patch = mask_mem_mito_ds & mask_geno_empty

print("V5-11 — mem-mito rows total:", int(mask_mem_mito_ds.sum()))
print("V5-11 — mem-mito rows needing patch:", int(mem_mito_to_patch.sum()))

# genotype: pDQM082 / 315
df_enrich.loc[mem_mito_to_patch, "genotype_base_codes"]   = "pDQM082"
df_enrich.loc[mem_mito_to_patch, "genotype_allele_codes"] = "315"

# treatment name for RNA: MGCO-01 (this will be mapped to a base code in V5-05)
mem_mito_treat_to_patch = mask_mem_mito_ds & mask_treat_rna_empty
print("V5-11 — mem-mito rows patched with MGCO-01 as treatment_rna_names_sheet:",
      int(mem_mito_treat_to_patch.sum()))
df_enrich.loc[mem_mito_treat_to_patch, "treatment_rna_names_sheet"] = "MGCO-01"

# ─────────────────────────────────────────────
# 2) Patch mem-histone datasets with no geno
#    Canonical mapping we agreed:
#      geno base codes:  pDQM005 | pDQM133
#      geno allele codes: 302 | 324
# ─────────────────────────────────────────────

mask_mem_histone_ds = df_enrich["dataset_slug"].astype(str).str.contains("mem-histone", case=False, na=False)
mem_histone_to_patch = mask_mem_histone_ds & mask_geno_empty

print("V5-11 — mem-histone rows total:", int(mask_mem_histone_ds.sum()))
print("V5-11 — mem-histone rows needing patch:", int(mem_histone_to_patch.sum()))

df_enrich.loc[mem_histone_to_patch, "genotype_base_codes"]   = "pDQM005|pDQM133"
df_enrich.loc[mem_histone_to_patch, "genotype_allele_codes"] = "302|324"

# ─────────────────────────────────────────────
# 3) Optional debug: quick peek at patched rows
# ─────────────────────────────────────────────

print("\nV5-11 — sample patched mem-mito rows:")
print(
    df_enrich.loc[mem_mito_to_patch,
                  ["roi_dir", "dataset_slug", "genotype_base_codes",
                   "genotype_allele_codes", "treatment_rna_names_sheet"]]
    .head(10)
)

print("\nV5-11 — sample patched mem-histone rows:")
print(
    df_enrich.loc[mem_histone_to_patch,
                  ["roi_dir", "dataset_slug", "genotype_base_codes",
                   "genotype_allele_codes"]]
    .head(10)
)

print("\nV5-11 — patch complete. Now re-run V5-05 → V5-08 (treatment mapping, markers, organelles, export).")

V5-11 — mem-mito rows total: 303
V5-11 — mem-mito rows needing patch: 2
V5-11 — mem-mito rows patched with MGCO-01 as treatment_rna_names_sheet: 303
V5-11 — mem-histone rows total: 141
V5-11 — mem-histone rows needing patch: 0

V5-11 — sample patched mem-mito rows:
                                               roi_dir          dataset_slug genotype_base_codes genotype_allele_codes treatment_rna_names_sheet
446  /clusterfs/vast/abcabc/Aang_Foundation/2025101...  20251016_mem-mito_v2             pDQM082                   315                   MGCO-01
447  /clusterfs/vast/abcabc/Aang_Foundation/2025101...  20251016_mem-mito_v2             pDQM082                   315                   MGCO-01

V5-11 — sample patched mem-histone rows:
Empty DataFrame
Columns: [roi_dir, dataset_slug, genotype_base_codes, genotype_allele_codes]
Index: []

V5-11 — patch complete. Now re-run V5-05 → V5-08 (treatment mapping, markers, organelles, export).


In [48]:
# V5-05 — attach treatment names (from imaging) and map to basecodes/organelle tags

import re
import numpy as np

# ─────────────────────────────────────────
# 1) Ensure canonical treatment_*_names_sheet columns exist on df_enrich
# ─────────────────────────────────────────

canonical_cols = {
    "treatment_plasmid_names_sheet": [
        "treatment_plasmid_names_sheet",
        "additional plasmids injected",
    ],
    "treatment_rna_names_sheet": [
        "treatment_rna_names_sheet",
        "additional mRNAs injected",
    ],
    "treatment_protein_names_sheet": [
        "treatment_protein_names_sheet",
        "additonal proteins injected",
    ],
    "treatment_dye_names_sheet": [
        "treatment_dye_names_sheet",
        "additonal dye and chemicals",
    ],
}

for canon, candidates in canonical_cols.items():
    if canon in df_enrich.columns:
        continue
    chosen = None
    for c in candidates:
        if c in df_enrich.columns:
            chosen = c
            break
    if chosen is not None:
        df_enrich = df_enrich.rename(columns={chosen: canon})
        print(f"V5-05 — renamed {chosen!r} → {canon!r}")
    else:
        df_enrich[canon] = np.nan
        print(f"V5-05 — WARNING: no source column found for {canon!r}; filled with NaN")

print("\nV5-05 — sample treatment_name columns from df_enrich:")
display(
    df_enrich[
        [
            "roi_dir",
            "treatment_plasmid_names_sheet",
            "treatment_rna_names_sheet",
            "treatment_protein_names_sheet",
            "treatment_dye_names_sheet",
        ]
    ].head(20)
)

# ─────────────────────────────────────────
# 2) Helpers
# ─────────────────────────────────────────

def _norm(s):
    if pd.isna(s):
        return None
    return str(s).strip()

def find_col(df: pd.DataFrame, required_substrings, optional_substrings=None):
    req = [r.lower() for r in required_substrings]
    opt = [o.lower() for o in (optional_substrings or [])]
    for col in df.columns:
        low = col.lower()
        if all(r in low for r in req) and all(o in low for o in opt):
            return col
    return None

# ─────────────────────────────────────────
# 3) Build plasmid / RNA mapping tables with guards
# ─────────────────────────────────────────

print("\nV5-05 — injected_plasmid columns:", list(injected_plasmid.columns))
print("V5-05 — injected_rna columns:", list(injected_rna.columns))

# ---- plasmid mapper ----
inj_pl = injected_plasmid.copy()

pl_name_col = (
    find_col(inj_pl, ["injection", "plasmid"])
    or find_col(inj_pl, ["plasmid", "name"])
    or find_col(inj_pl, ["plasmid"])
)
pl_base_col = (
    find_col(inj_pl, ["plasmid", "base"])
    or find_col(inj_pl, ["plasmid", "code"])
)

pl_fluor_col = find_col(inj_pl, ["fluor"])
pl_tag_col   = find_col(inj_pl, ["tag"])
pl_loc_col   = find_col(inj_pl, ["localization"]) or find_col(inj_pl, ["organelle"])

injected_plasmid_small = None
if pl_name_col and pl_base_col:
    injected_plasmid_small = inj_pl.rename(
        columns={
            pl_name_col: "treatment_name",
            pl_base_col: "plasmid_base_code",
            **({pl_fluor_col: "fluor_code"} if pl_fluor_col else {}),
            **({pl_tag_col: "tag_code"} if pl_tag_col else {}),
            **({pl_loc_col: "tag_localization"} if pl_loc_col else {}),
        }
    )
    if "treatment_name" in injected_plasmid_small.columns:
        injected_plasmid_small["treatment_name_norm"] = injected_plasmid_small["treatment_name"].apply(_norm)
    else:
        print("\nV5-05 — WARNING: plasmid rename did not produce 'treatment_name'; disabling plasmid mapper.")
        print("           columns:", list(injected_plasmid_small.columns))
        injected_plasmid_small = None
else:
    print("\nV5-05 — WARNING: could not detect name/basecode columns in injected_plasmid; plasmid treatments will not be mapped.")

# ---- RNA mapper ----
inj_rna = injected_rna.copy()

# Special-case: your injected_rna sheet has exactly these cols:
#   injected_rna (name) + plasmid_base_code (basecode for the RNA construct)
if "injected_rna" in inj_rna.columns:
    rna_name_col = "injected_rna"
else:
    rna_name_col = (
        find_col(inj_rna, ["injection", "rna"])
        or find_col(inj_rna, ["rna", "name"])
        or find_col(inj_rna, ["rna"])
    )

if "plasmid_base_code" in inj_rna.columns:
    # Even though it says "plasmid", for this sheet it's the basecode of the RNA construct
    rna_base_col = "plasmid_base_code"
else:
    rna_base_col = (
        find_col(inj_rna, ["rna", "base"])
        or find_col(inj_rna, ["rna", "code"])
    )

rna_fluor_col = find_col(inj_rna, ["fluor"])
rna_tag_col   = find_col(inj_rna, ["tag"])
rna_loc_col   = find_col(inj_rna, ["localization"]) or find_col(inj_rna, ["organelle"])

injected_rna_small = None
if rna_name_col and rna_base_col:
    injected_rna_small = inj_rna.rename(
        columns={
            rna_name_col: "treatment_name",
            rna_base_col: "rna_base_code",
            **({rna_fluor_col: "fluor_code"} if rna_fluor_col else {}),
            **({rna_tag_col: "tag_code"} if rna_tag_col else {}),
            **({rna_loc_col: "tag_localization"} if rna_loc_col else {}),
        }
    )
    if "treatment_name" in injected_rna_small.columns:
        injected_rna_small["treatment_name_norm"] = injected_rna_small["treatment_name"].apply(_norm)
    else:
        print("\nV5-05 — WARNING: RNA rename did not produce 'treatment_name'; disabling RNA mapper.")
        print("           columns:", list(injected_rna_small.columns))
        injected_rna_small = None
else:
    print("\nV5-05 — WARNING: could not detect name/basecode columns in injected_rna; RNA treatments will not be mapped.")

# ─────────────────────────────────────────
# 4) Map treatment_*_names_sheet → basecodes + marker tags
# ─────────────────────────────────────────

def split_names(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    parts = re.split(r"[;,|]", s)
    return [p.strip() for p in parts if p.strip()]

def uniq_pipe(vals):
    vals = [str(x).strip() for x in vals if pd.notna(x) and str(x).strip()]
    if not vals:
        return np.nan
    seen = []
    for v in vals:
        if v not in seen:
            seen.append(v)
    return "|".join(seen)

def map_treatment_kind(
    df: pd.DataFrame,
    sheet_col: str,
    mapper_df: pd.DataFrame | None,
    kind: str,
) -> pd.DataFrame:
    if mapper_df is None or sheet_col not in df.columns:
        return df

    out = df.copy()

    exploded = out[["roi_dir", sheet_col]].copy()
    exploded[f"{sheet_col}_list"] = exploded[sheet_col].apply(split_names)
    exploded = exploded.explode(f"{sheet_col}_list")
    exploded = exploded.rename(columns={f"{sheet_col}_list": "treatment_name"})
    exploded["treatment_name_norm"] = exploded["treatment_name"].apply(_norm)

    merged = exploded.merge(
        mapper_df,
        on="treatment_name_norm",
        how="left",
        suffixes=("", "_map"),
    )

    if kind == "plasmid":
        code_col = "plasmid_base_code"
        out_code   = "treatment_plasmid_plasmid_base_code"
        out_fluor  = "treatment_plasmid_fluor_code"
        out_tag    = "treatment_plasmid_tag_code"
        out_loc    = "treatment_plasmid_tag_localization"
    else:
        code_col = "rna_base_code"
        out_code   = "treatment_rna_rna_base_code"
        out_fluor  = "treatment_rna_fluor_code"
        out_tag    = "treatment_rna_tag_code"
        out_loc    = "treatment_rna_tag_localization"

    cols_present = {
        "code": code_col in merged.columns,
        "fluor": "fluor_code" in merged.columns,
        "tag": "tag_code" in merged.columns,
        "loc": "tag_localization" in merged.columns,
    }

    if not any(cols_present.values()):
        return out

    agg_dict = {}
    if cols_present["code"]:
        agg_dict[out_code] = (code_col, uniq_pipe)
    if cols_present["fluor"]:
        agg_dict[out_fluor] = ("fluor_code", uniq_pipe)
    if cols_present["tag"]:
        agg_dict[out_tag] = ("tag_code", uniq_pipe)
    if cols_present["loc"]:
        agg_dict[out_loc] = ("tag_localization", uniq_pipe)

    if not agg_dict:
        return out

    agg = (
        merged
        .groupby("roi_dir")
        .agg(**{new: (src, func) for new, (src, func) in agg_dict.items()})
        .reset_index()
    )

    out = out.merge(agg, on="roi_dir", how="left")
    return out

df_enrich = map_treatment_kind(
    df_enrich,
    sheet_col="treatment_plasmid_names_sheet",
    mapper_df=injected_plasmid_small,
    kind="plasmid",
)

df_enrich = map_treatment_kind(
    df_enrich,
    sheet_col="treatment_rna_names_sheet",
    mapper_df=injected_rna_small,
    kind="rna",
)

print("\nV5-05 — df_enrich after treatment mapping, sample:")
cols_show = [
    "roi_dir",
    "treatment_plasmid_names_sheet",
    "treatment_plasmid_plasmid_base_code",
    "treatment_plasmid_fluor_code",
    "treatment_plasmid_tag_code",
    "treatment_plasmid_tag_localization",
    "treatment_rna_names_sheet",
    "treatment_rna_rna_base_code",
    "treatment_rna_fluor_code",
    "treatment_rna_tag_code",
    "treatment_rna_tag_localization",
]
cols_show = [c for c in cols_show if c in df_enrich.columns]
display(df_enrich[cols_show].head(20))

V5-05 — renamed 'additional plasmids injected' → 'treatment_plasmid_names_sheet'
V5-05 — renamed 'additonal proteins injected' → 'treatment_protein_names_sheet'
V5-05 — renamed 'additonal dye and chemicals' → 'treatment_dye_names_sheet'

V5-05 — sample treatment_name columns from df_enrich:


,roi_dir,treatment_plasmid_names_sheet,treatment_rna_names_sheet,treatment_protein_names_sheet,treatment_dye_names_sheet
0,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
1,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
2,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
3,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
4,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
5,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,JF 635
6,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,JF 635
7,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,JF 635
8,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,JF 635
9,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,JF 635



V5-05 — injected_plasmid columns: ['injected_plasmid', 'plasmid_base_code']
V5-05 — injected_rna columns: ['injected_rna', 'plasmid_base_code']

V5-05 — df_enrich after treatment mapping, sample:


,roi_dir,treatment_plasmid_names_sheet,treatment_plasmid_plasmid_base_code,treatment_rna_names_sheet,treatment_rna_rna_base_code
0,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
1,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
2,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
3,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
4,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
5,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
6,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
7,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
8,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN
9,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN,NaN,NaN,NaN


In [49]:
# V5-06 — canonical organelles and fluor-organelle rollup

def split_pipe(val):
    if pd.isna(val) or not isinstance(val, str) or not val.strip():
        return []
    return [x.strip() for x in val.split("|") if x.strip()]

def join_unique(items):
    out = []
    seen = set()
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return "|".join(out) if out else np.nan

geno_locs  = df_enrich.get("genotype_marker_localizations")
treat_pl_locs = df_enrich.get("treatment_plasmid_tag_localization")
treat_rna_locs = df_enrich.get("treatment_rna_tag_localization")

all_unique = []
all_fluor  = []

for idx in range(len(df_enrich)):
    loc_items = []
    fluor_items = []

    if geno_locs is not None:
        loc_items += split_pipe(geno_locs.iloc[idx])

    if treat_pl_locs is not None:
        loc_items += split_pipe(treat_pl_locs.iloc[idx])
    if treat_rna_locs is not None:
        loc_items += split_pipe(treat_rna_locs.iloc[idx])

    all_unique.append(join_unique(loc_items))

    geno_fluors = split_pipe(df_enrich.get("genotype_marker_fluor_codes", pd.Series([np.nan]*len(df_enrich))).iloc[idx])
    geno_loc    = split_pipe(df_enrich.get("genotype_marker_localizations", pd.Series([np.nan]*len(df_enrich))).iloc[idx])

    if geno_fluors and geno_loc:
        for f in geno_fluors:
            for loc in geno_loc:
                fluor_items.append(f"{f}({loc})")

    pl_f = split_pipe(df_enrich.get("treatment_plasmid_fluor_code", pd.Series([np.nan]*len(df_enrich))).iloc[idx])
    pl_l = split_pipe(df_enrich.get("treatment_plasmid_tag_localization", pd.Series([np.nan]*len(df_enrich))).iloc[idx])
    if pl_f and pl_l:
        for f in pl_f:
            for loc in pl_l:
                fluor_items.append(f"{f}({loc})")

    rna_f = split_pipe(df_enrich.get("treatment_rna_fluor_code", pd.Series([np.nan]*len(df_enrich))).iloc[idx])
    rna_l = split_pipe(df_enrich.get("treatment_rna_tag_localization", pd.Series([np.nan]*len(df_enrich))).iloc[idx])
    if rna_f and rna_l:
        for f in rna_f:
            for loc in rna_l:
                fluor_items.append(f"{f}({loc})")

    all_fluor.append(join_unique(fluor_items))

df_enrich["all_unique_organelles"] = all_unique
df_enrich["all_fluor_organelles"]  = all_fluor

print("\nV5-06 — organelle coverage:")
print(df_enrich["all_unique_organelles"].notna().value_counts())

print("\nV5-06 — sample rows with organelles:")
display(
    df_enrich.loc[
        df_enrich["all_unique_organelles"].notna(),
        [
            "roi_dir",
            "genotype_marker_fluor_codes",
            "genotype_marker_localizations",
            "treatment_plasmid_names_sheet",
            "treatment_rna_names_sheet",
            "all_unique_organelles",
            "all_fluor_organelles",
        ],
    ].head(30)
)


V5-06 — organelle coverage:
all_unique_organelles
True     688
False    288
Name: count, dtype: int64

V5-06 — sample rows with organelles:


,roi_dir,genotype_marker_fluor_codes,genotype_marker_localizations,treatment_plasmid_names_sheet,treatment_rna_names_sheet,all_unique_organelles,all_fluor_organelles
14,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
15,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
16,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
17,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
18,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
19,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
20,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
21,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
22,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)
23,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,tdmChilada,membrane,NaN,NaN,membrane,tdmChilada(membrane)


In [50]:
# V5-07 — build df_for_db subset and write outputs

required_cols = [
    "roi_dir",
    "bruker_roi_id",
    "plate_date",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "dataset_slug",        # if not present, dataset_slug_norm is fallback
    "experiment_folder",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "roi_anatomy",
    "roi_tiffs",
    "date_experiment",
    "Date imaged",
    "date_mount",
    "parent_female_name",
    "parent_male_name",
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_plasmid_names_sheet",
    "treatment_rna_names_sheet",
    "treatment_plasmid_plasmid_base_code",
    "treatment_rna_rna_base_code",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "genotype_marker_fusion_labels",
    "all_unique_organelles",
    "all_fluor_organelles",
    "link_source",
]

# dataset_slug fallback
if "dataset_slug" not in df_enrich.columns and "dataset_slug_norm" in df_enrich.columns:
    df_enrich["dataset_slug"] = df_enrich["dataset_slug_norm"]

missing_req = [c for c in required_cols if c not in df_enrich.columns]
if missing_req:
    print("V5-07 — WARNING: missing some required columns; they will be skipped:", missing_req)

present_req = [c for c in required_cols if c in df_enrich.columns]

df_for_db = df_enrich[present_req].copy()

print("\nV5-07 — df_for_db shape:", df_for_db.shape)
print("V5-07 — df_for_db columns:")
print(list(df_for_db.columns))

FULL_OUT = WORKING / "legacy_imaging_annotations_v5.csv"
DB_OUT   = WORKING / "legacy_imaging_annotations_for_db_v5.csv"

df_enrich.to_csv(FULL_OUT, index=False)
df_for_db.to_csv(DB_OUT, index=False)

print("\nV5-07 — wrote full annotations to:", FULL_OUT)
print("V5-07 — wrote DB subset to:      ", DB_OUT)

print("\nV5-07 — DB subset sample:")
display(df_for_db.head(20))

print("\nV5-07 — DB subset organelle coverage:")
print(df_for_db["all_unique_organelles"].notna().value_counts(dropna=False))


V5-07 — df_for_db shape: (976, 31)
V5-07 — df_for_db columns:
['roi_dir', 'bruker_roi_id', 'plate_date', 'plate_id_filled', 'slot_id_filled', 'roi_index_within_slot', 'dataset_slug', 'experiment_folder', 'fish_id', 'fish_number', 'fish_age_hpf', 'roi_anatomy', 'roi_tiffs', 'date_experiment', 'Date imaged', 'date_mount', 'parent_female_name', 'parent_male_name', 'genotype_base_codes', 'genotype_allele_codes', 'treatment_plasmid_names_sheet', 'treatment_rna_names_sheet', 'treatment_plasmid_plasmid_base_code', 'treatment_rna_rna_base_code', 'genotype_marker_fluor_codes', 'genotype_marker_tag_codes', 'genotype_marker_localizations', 'genotype_marker_fusion_labels', 'all_unique_organelles', 'all_fluor_organelles', 'link_source']

V5-07 — wrote full annotations to: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_v5.csv
V5-07 — wrote DB subset to:       /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_an

,roi_dir,bruker_roi_id,plate_date,plate_id_filled,slot_id_filled,roi_index_within_slot,dataset_slug,experiment_folder,fish_id,fish_number,fish_age_hpf,roi_anatomy,roi_tiffs,date_experiment,Date imaged,date_mount,parent_female_name,parent_male_name,genotype_base_codes,genotype_allele_codes,treatment_plasmid_names_sheet,treatment_rna_names_sheet,treatment_plasmid_plasmid_base_code,treatment_rna_rna_base_code,genotype_marker_fluor_codes,genotype_marker_tag_codes,genotype_marker_localizations,genotype_marker_fusion_labels,all_unique_organelles,all_fluor_organelles,link_source
0,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250721-plate65-slot1-roi1,20250721.0,65.0,1.0,1.0,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,fish1,1.0,NaN,NaN,74,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
1,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250721-plate65-slot2-roi1,20250721.0,65.0,2.0,1.0,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,fish2,2.0,NaN,NaN,74,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
2,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250721-plate65-slot3-roi1,20250721.0,65.0,3.0,1.0,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,fish3,3.0,NaN,NaN,80,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
3,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250721-plate65-slot1-roi2,20250721.0,65.0,1.0,2.0,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,fish1,1.0,NaN,NaN,50,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
4,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250721-plate65-slot2-roi2,20250721.0,65.0,2.0,2.0,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,fish2,2.0,NaN,NaN,80,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
5,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250722-plate66-slot1-roi1,20250722.0,66.0,1.0,1.0,20250722_mem-halo_er-mSG,20250722_mem-halo_er-mSG,fish1,1.0,24.0,NaN,33,20250722_mem-halo_er-mSG,2025-07-22,2025-07-22,membrane Halo,membrane Halo,pSWIN01,is01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sheet
6,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250722-plate66-slot1-roi2,20250722.0,66.0,1.0,2.0,20250722_mem-halo_er-mSG,20250722_mem-halo_er-mSG,fish1,1.0,24.0,NaN,109,20250722_mem-halo_er-mSG,2025-07-22,2025-07-22,membrane Halo,membrane Halo,pSWIN01,is01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sheet
7,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250722-plate66-slot1-roi3,20250722.0,66.0,1.0,3.0,20250722_mem-halo_er-mSG,20250722_mem-halo_er-mSG,fish1,1.0,24.0,NaN,37,20250722_mem-halo_er-mSG,2025-07-22,2025-07-22,membrane Halo,membrane Halo,pSWIN01,is01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sheet
8,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250722-plate66-slot1-roi4,20250722.0,66.0,1.0,4.0,20250722_mem-halo_er-mSG,20250722_mem-halo_er-mSG,fish1,1.0,24.0,NaN,140,20250722_mem-halo_er-mSG,2025-07-22,2025-07-22,membrane Halo,membrane Halo,pSWIN01,is01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sheet
9,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,20250722-plate66-slot1-roi5,20250722.0,66.0,1.0,5.0,20250722_mem-halo_er-mSG,20250722_mem-halo_er-mSG,fish1,1.0,24.0,NaN,37,20250722_mem-halo_er-mSG,2025-07-22,2025-07-22,membrane Halo,membrane Halo,pSWIN01,is01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sheet



V5-07 — DB subset organelle coverage:
all_unique_organelles
True     688
False    288
Name: count, dtype: int64


In [51]:
# V5-07b — organelle coverage QC (robust to column-name variants)

df_org = df_for_db.copy()

mask_no_org = df_org["all_unique_organelles"].isna() | (df_org["all_unique_organelles"].astype(str).str.strip() == "")
n_total = len(df_org)
n_no_org = mask_no_org.sum()

print("V5-07b — organelle coverage QC")
print(f"  total rows in df_for_db: {n_total}")
print(f"  ROIs with all_unique_organelles = NULL/empty: {n_no_org}")

missing = df_org[mask_no_org]

print("\nV5-07b — breakdown by link_source:")
print(missing["link_source"].value_counts(dropna=False))

print("\nV5-07b — breakdown by genotype presence (base codes):")
has_genotype = (
    missing["genotype_base_codes"].notna()
    & (missing["genotype_base_codes"].astype(str).str.strip() != "")
)
print(has_genotype.value_counts(dropna=False).rename("has_genotype"))

# ---- treatment presence (plasmid or RNA) ----

# try to find plausible treatment base-code columns
plasmid_base_cols = [c for c in df_org.columns if "treatment_plasmid" in c and "base_code" in c]
rna_base_cols      = [c for c in df_org.columns if "treatment_rna"     in c and "base_code" in c]

print("\nV5-07b — treatment base-code columns detected:")
print("  plasmid_base_cols:", plasmid_base_cols)
print("  rna_base_cols:    ", rna_base_cols)

def _has_any_nonempty(row, cols):
    for c in cols:
        val = row.get(c)
        if pd.notna(val) and str(val).strip() != "":
            return True
    return False

if plasmid_base_cols or rna_base_cols:
    missing = missing.copy()
    missing["has_treatment"] = missing.apply(
        lambda r: _has_any_nonempty(r, plasmid_base_cols + rna_base_cols),
        axis=1,
    )
    print("\nV5-07b — breakdown by treatment presence (plasmid or RNA):")
    print(missing["has_treatment"].value_counts(dropna=False))
else:
    print("\nV5-07b — no treatment base-code columns detected; skipping has_treatment summary.")

print("\nV5-07b — top datasets among no-org rows:")
print(
    missing["dataset_slug"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nV5-07b — sample no-org rows:")
cols_show = [
    "roi_dir",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "roi_anatomy",
    "genotype_base_codes",
]
cols_show += plasmid_base_cols + rna_base_cols + [
    "genotype_marker_fluor_codes",
    "genotype_marker_localizations",
    "all_unique_organelles",
    "all_fluor_organelles",
    "link_source",
]
cols_show = [c for c in cols_show if c in missing.columns]

print(
    missing[cols_show]
    .head(30)
    .to_string(index=False)
)

V5-07b — organelle coverage QC
  total rows in df_for_db: 976
  ROIs with all_unique_organelles = NULL/empty: 288

V5-07b — breakdown by link_source:
link_source
sheet         170
unmatched     112
date_match      6
Name: count, dtype: int64

V5-07b — breakdown by genotype presence (base codes):
genotype_base_codes
True     253
False     35
Name: has_genotype, dtype: int64

V5-07b — treatment base-code columns detected:
  plasmid_base_cols: ['treatment_plasmid_plasmid_base_code']
  rna_base_cols:     ['treatment_rna_rna_base_code']

V5-07b — breakdown by treatment presence (plasmid or RNA):
has_treatment
False    288
Name: count, dtype: int64

V5-07b — top datasets among no-org rows:
dataset_slug
20250513_skittles                            44
20251002_mem_mito                            24
20250917_mem-mito                            21
20250522_skittlez                            19
20250915_mem-mito                            18
20250924_mem-mito                            18
202511

In [52]:
# V5-07c — push treatment RNAs into marker/organelle rollup via constructs_ft

import pandas as pd

print("\nV5-07c — start treatment-RNA marker plumbing")

# ─────────────────────────────────────────────
# 0. Preconditions
# ─────────────────────────────────────────────
if "treatment_rna_rna_base_code" not in df_enrich.columns:
    raise KeyError("V5-07c: df_enrich missing 'treatment_rna_rna_base_code'; run V5-05 first.")

required_geno_cols = [
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "all_unique_organelles",
    "all_fluor_organelles",
]
missing_geno = [c for c in required_geno_cols if c not in df_enrich.columns]
if missing_geno:
    raise KeyError(f"V5-07c: df_enrich missing expected genotype marker columns: {missing_geno}")

if "roi_dir" not in df_enrich.columns:
    raise KeyError("V5-07c: df_enrich missing 'roi_dir' column.")


# ─────────────────────────────────────────────
# 1. (Re)build constructs_ft (plasmid → fluor/tag/localization)
# ─────────────────────────────────────────────
def _build_constructs_ft(constructs: pd.DataFrame,
                         fluors: pd.DataFrame,
                         tags: pd.DataFrame) -> pd.DataFrame:
    # normalize constructs column names
    c = constructs.rename(columns={"code": "plasmid_code"})
    # keep minimal, one row per plasmid/fluor/tag
    if "plasmid_code" not in c.columns:
        raise KeyError("constructs is missing 'plasmid_code' or 'code' column.")

    # Try to identify fluor/tag columns in constructs
    fluor_col = None
    for cand in ["fluor_code", "fluor", "fluor_name"]:
        if cand in c.columns:
            fluor_col = cand
            break

    tag_col = None
    for cand in ["tag_code", "tag", "tag_name"]:
        if cand in c.columns:
            tag_col = cand
            break

    # If constructs already has fluor/tag/localization, use those directly
    cols = ["plasmid_code"]
    if fluor_col:
        cols.append(fluor_col)
    if tag_col:
        cols.append(tag_col)

    constructs_small = c[cols].copy().drop_duplicates()

    # bring in fluor nm/etc if available, but not strictly required here
    if "nickname" in fluors.columns:
        fluors_small = fluors.rename(columns={"nickname": "fluor_code"})
    else:
        fluors_small = fluors.copy()
        if "fluor_code" not in fluors_small.columns:
            fluors_small["fluor_code"] = fluors_small.iloc[:, 0]

    # tags: we really only need localization
    if "nickname" in tags.columns:
        tags_small = tags.rename(columns={"nickname": "tag_code"})
    else:
        tags_small = tags.copy()
        if "tag_code" not in tags_small.columns:
            tags_small["tag_code"] = tags_small.iloc[:, 0]

    if "localization" in tags_small.columns:
        tags_small = tags_small[["tag_code", "localization"]].rename(
            columns={"localization": "tag_localization"}
        )
    else:
        # default to cytosol for unknown tags
        tags_small = tags_small[["tag_code"]].copy()
        tags_small["tag_localization"] = "cytosol"

    # join fluor metadata (if present)
    if "fluor_code" in constructs_small.columns and "fluor_code" in fluors_small.columns:
        constructs_ft = constructs_small.merge(
            fluors_small[["fluor_code"]].drop_duplicates(),
            on="fluor_code",
            how="left",
        )
    else:
        constructs_ft = constructs_small.copy()
        if "fluor_code" not in constructs_ft.columns:
            constructs_ft["fluor_code"] = pd.NA

    # join tag localization
    if "tag_code" in constructs_ft.columns:
        constructs_ft = constructs_ft.merge(
            tags_small,
            on="tag_code",
            how="left",
        )
    else:
        constructs_ft["tag_code"] = pd.NA
        constructs_ft["tag_localization"] = pd.NA

    # fill missing localizations for untagged constructs with cytosol
    constructs_ft["tag_localization"] = constructs_ft["tag_localization"].fillna("cytosol")

    return constructs_ft[["plasmid_code", "fluor_code", "tag_code", "tag_localization"]].drop_duplicates()


try:
    constructs_ft
    if not isinstance(constructs_ft, pd.DataFrame):
        raise NameError
    print("V5-07c — using existing constructs_ft")
except NameError:
    print("V5-07c — rebuilding constructs_ft from constructs/fluors/tags")
    if "constructs" not in globals() or "fluors" not in globals() or "tags" not in globals():
        raise KeyError("V5-07c: constructs/fluors/tags DataFrames are not available in this notebook.")
    constructs_ft = _build_constructs_ft(constructs, fluors, tags)

print("V5-07c — constructs_ft sample:")
print(constructs_ft.head(10))


# ─────────────────────────────────────────────
# 2. Build treatment RNA marker columns via constructs_ft
# ─────────────────────────────────────────────
def _split_pipe(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    return [x.strip() for x in s.split("|") if x.strip()]

# explode treatment RNA basecodes
tx_rna = df_enrich[["roi_dir", "treatment_rna_rna_base_code"]].copy()
tx_rna["treatment_rna_rna_base_code"] = tx_rna["treatment_rna_rna_base_code"].fillna("")

tx_rna = tx_rna.loc[tx_rna["treatment_rna_rna_base_code"].astype(str).str.strip() != ""].copy()
if tx_rna.empty:
    print("\nV5-07c — no treatment_rna_rna_base_code values found; keeping existing organelles unchanged.")
else:
    tx_rna["rna_base_code"] = tx_rna["treatment_rna_rna_base_code"].apply(
        lambda s: _split_pipe(s)[0] if isinstance(s, str) else None
    )
    tx_rna = tx_rna.loc[tx_rna["rna_base_code"].notna() & (tx_rna["rna_base_code"].astype(str).str.strip() != "")]
    print(f"\nV5-07c — treatment RNA rows with basecodes: {len(tx_rna)}")

    # join to constructs_ft: rna_base_code behaves like plasmid_code
    tx_rna_markers = tx_rna.merge(
        constructs_ft.rename(columns={"plasmid_code": "rna_base_code"}),
        on="rna_base_code",
        how="left",
    )

    print("V5-07c — tx_rna_markers sample:")
    print(tx_rna_markers.head(10))

    def _agg_tx(group: pd.DataFrame) -> pd.Series:
        f = [x for x in group["fluor_code"].astype(str) if x and x != "nan"]
        t = [x for x in group["tag_code"].astype(str) if x and x != "nan"]
        loc = [x for x in group["tag_localization"].astype(str) if x and x != "nan"]

        def _uniq(xs):
            out = []
            seen = set()
            for v in xs:
                if v not in seen:
                    seen.add(v)
                    out.append(v)
            return out

        f_u = _uniq(f)
        t_u = _uniq(t)
        loc_u = _uniq(loc)

        # fluor_loc labels like mScarlet3(histone)
        fluor_loc = []
        for ff, ll in zip(f_u, loc_u):
            if ff and ll:
                fluor_loc.append(f"{ff}({ll})")
        fluor_loc = _uniq(fluor_loc)

        return pd.Series(
            {
                "treatment_marker_fluor_codes": "|".join(f_u) if f_u else pd.NA,
                "treatment_marker_tag_codes": "|".join(t_u) if t_u else pd.NA,
                "treatment_marker_localizations": "|".join(loc_u) if loc_u else pd.NA,
                "treatment_marker_fluor_loc_labels": "|".join(fluor_loc) if fluor_loc else pd.NA,
            }
        )

    tx_rna_agg = tx_rna_markers.groupby("roi_dir", as_index=False).apply(_agg_tx)
    print("\nV5-07c — tx_rna_agg sample:")
    print(tx_rna_agg.head(10))

    # merge back to df_enrich (add / overwrite treatment_marker_* columns)
    df_enrich = df_enrich.merge(tx_rna_agg, on="roi_dir", how="left")


# ─────────────────────────────────────────────
# 3. Recompute all_unique_organelles / all_fluor_organelles
#    as union(genotype + treatment)
# ─────────────────────────────────────────────
def _split_to_list(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    return [x.strip() for x in s.split("|") if x.strip()]

def _join_unique(vals):
    out = []
    seen = set()
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return "|".join(out) if out else pd.NA

geno_locs = df_enrich["genotype_marker_localizations"].apply(_split_to_list)
geno_fl   = df_enrich["genotype_marker_fluor_codes"].apply(_split_to_list)
geno_florg = df_enrich["all_fluor_organelles"].apply(_split_to_list)

tx_locs = df_enrich.get("treatment_marker_localizations", pd.Series([pd.NA] * len(df_enrich))).apply(_split_to_list)
tx_fl   = df_enrich.get("treatment_marker_fluor_codes", pd.Series([pd.NA] * len(df_enrich))).apply(_split_to_list)
tx_florg = df_enrich.get("treatment_marker_fluor_loc_labels", pd.Series([pd.NA] * len(df_enrich))).apply(_split_to_list)

all_unique = []
all_florg  = []

for gl, tl, gf, tf, gf_org, tf_org in zip(geno_locs, tx_locs, geno_fl, tx_fl, geno_florg, tx_florg):
    # organelles: union of genotype + treatment localizations
    locs = _join_unique(gl + tl)

    # fluor organelles: union of existing genotype fluor_loc labels + tx fluor_loc labels
    florg = _join_unique(gf_org + tf_org)

    all_unique.append(locs)
    all_florg.append(florg)

df_enrich["all_unique_organelles"] = all_unique
df_enrich["all_fluor_organelles"]  = all_florg

print("\nV5-07c — recomputed organelles:")
print(df_enrich[["roi_dir", "genotype_marker_localizations",
                 "treatment_marker_localizations",
                 "all_unique_organelles",
                 "all_fluor_organelles"]].head(20))

# small coverage check
mask_missing = df_enrich["all_unique_organelles"].isna()
print("\nV5-07c — organelle coverage after union:")
print("  total rows:", len(df_enrich))
print("  missing all_unique_organelles:", mask_missing.sum())
print("  by dataset_slug (top 10):")
print(df_enrich.loc[mask_missing, "dataset_slug"].value_counts().head(10))


V5-07c — start treatment-RNA marker plumbing
V5-07c — using existing constructs_ft
V5-07c — constructs_ft sample:
  plasmid_code            plasmid_name        plasmid_nickname resistance                                      plasmid_notes  used_for_injection_plasmid used_for_injection_rna  used_for_injection_crispr  n_fluors_per_plasmid fluor_code  \
0      pDQM001  CMV-SP6-mSG(J) IDT opt  CMV-SP6-mSG(J) IDT opt        Amp  IDT optimized, for mRNA production with SP6 - ...                           1                  FALSE                      False                   1.0        mSG   
1      pDQM002  CMV-SP6-mSG(J) IDT opt  CMV-SP6-mSG(J) IDT opt        Amp  IDT optimized, for mRNA production with SP6 - ...                           1                  FALSE                      False                   1.0        mSG   
2      pDQM005           ef1a-tdmSG(J)           ef1a-tdmSG(J)        Amp     iCodon optimized for tol2 insertions - clone 4                           1                

KeyError: "['treatment_marker_localizations'] not in index"

In [53]:
# V5-07d (v2) — collapse duplicate roi_dir rows by aggregating metadata

import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V5-07d (v2): df_enrich not found; run earlier V5 cells first.")

print("V5-07d (v2) — starting dedupe on roi_dir")
print("  df_enrich shape BEFORE:", df_enrich.shape)
print("  unique roi_dir BEFORE:", df_enrich["roi_dir"].nunique())

# quick scan of duplicates
dups = df_enrich[df_enrich.duplicated("roi_dir", keep=False)].copy()
print(f"\nV5-07d (v2) — duplicate roi_dir rows: {len(dups)}")
print(f"V5-07d (v2) — duplicate roi_dir keys: {dups['roi_dir'].nunique()}")

if len(dups) == 0:
    print("V5-07d (v2) — no duplicates; nothing to do.")
else:
    # columns that are pure linkage / indexing and can differ freely
    linkage_cols = {
        "bruker_roi_id",
        "plate_id_filled",
        "slot_id_filled",
        "roi_index_within_slot",
        "mount_id_inferred",
        "plate_key",
    }

    all_cols = list(df_enrich.columns)

    def _agg_group(g: pd.DataFrame) -> pd.Series:
        out = {}
        for col in all_cols:
            if col == "roi_dir":
                continue
            vals = g[col]

            # allow linkage cols to just take the first value
            if col in linkage_cols:
                out[col] = vals.iloc[0]
                continue

            # if everything is NA / empty
            non_null = vals.dropna()
            if non_null.empty:
                out[col] = pd.NA
                continue

            # numeric-ish columns: if multiple distinct, keep the first
            if pd.api.types.is_numeric_dtype(vals):
                uniq = non_null.unique()
                out[col] = uniq[0]
                continue

            # string / mixed: union unique non-empty values
            uniq = sorted(
                set(
                    s.strip()
                    for s in non_null.astype(str)
                    if s is not None and str(s).strip() != ""
                )
            )
            if not uniq:
                out[col] = pd.NA
            elif len(uniq) == 1:
                out[col] = uniq[0]
            else:
                # deterministic union in sorted order
                out[col] = "|".join(uniq)

            # special case: if this is one of the sheet-only text fields
            # like 'Unique Targets with blanks' / 'Unique Targets' / 'treatment_rna_names_sheet',
            # this union behavior is exactly what we want.
        return pd.Series(out, index=[c for c in all_cols if c != "roi_dir"])

    # apply aggregator per roi_dir
    df_enrich = (
        df_enrich
        .groupby("roi_dir", as_index=False)
        .apply(_agg_group)
        .reset_index(drop=True)
    )

    print("\nV5-07d (v2) — dedupe complete.")
    print("  df_enrich shape AFTER:", df_enrich.shape)
    print("  unique roi_dir AFTER:", df_enrich["roi_dir"].nunique())

    # sanity check: no remaining duplicates
    remaining_dups = df_enrich[df_enrich.duplicated("roi_dir", keep=False)]
    print("  remaining duplicate roi_dir rows:", len(remaining_dups))
    if len(remaining_dups):
        print("  WARNING: some roi_dir still duplicated; inspect manually.")

V5-07d (v2) — starting dedupe on roi_dir
  df_enrich shape BEFORE: (976, 69)
  unique roi_dir BEFORE: 976

V5-07d (v2) — duplicate roi_dir rows: 0
V5-07d (v2) — duplicate roi_dir keys: 0
V5-07d (v2) — no duplicates; nothing to do.


In [54]:
# V5-08 — build final DB subset and write v5 CSVs

from pathlib import Path
import pandas as pd

# ─────────────────────────────────────────────
# 1) Define output paths
# ─────────────────────────────────────────────
ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
WORKING = BASE / "working"
FINAL   = BASE / "final"

for p in [WORKING, FINAL]:
    p.mkdir(parents=True, exist_ok=True)

FULL_OUT = WORKING / "legacy_imaging_annotations_v5.csv"
DB_OUT   = WORKING / "legacy_imaging_annotations_for_db_v5.csv"

print("V5-08 — FULL_OUT:", FULL_OUT)
print("V5-08 — DB_OUT:  ", DB_OUT)

# ─────────────────────────────────────────────
# 2) Start from df_enrich and sanity check
# ─────────────────────────────────────────────
if "df_enrich" not in globals():
    raise NameError("V5-08: df_enrich not found; run earlier V5 cells first.")

print("V5-08 — df_enrich shape:", df_enrich.shape)

# ─────────────────────────────────────────────
# 3) Define canonical DB subset columns
#    (only keep columns that actually exist)
# ─────────────────────────────────────────────
candidate_cols = [
    # identity / hierarchy
    "roi_dir",
    "bruker_roi_id",
    "plate_date",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "roi_anatomy",
    "roi_tiffs",

    # timing / sheet metadata
    "date_experiment",
    "Date imaged",
    "date_mount",

    # genetics
    "ZF female genotype",
    "ZF male genotype",
    "genotype_base_codes",
    "genotype_allele_codes",

    # treatments (base codes from mappers)
    "treatment_plasmid_plasmid_base_code",
    "treatment_rna_rna_base_code",

    # genotype markers
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "genotype_marker_fusion_labels",

    # treatment markers (from RNAs / plasmids, if present)
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "treatment_marker_localizations",
    "treatment_marker_fluor_loc_labels",

    # union organelles
    "all_unique_organelles",
    "all_fluor_organelles",

    # provenance
    "link_source",
]

db_cols = [c for c in candidate_cols if c in df_enrich.columns]
missing_cols = [c for c in candidate_cols if c not in df_enrich.columns]

print("\nV5-08 — DB column selection:")
print("  kept:", db_cols)
print("  missing (not in df_enrich):", missing_cols)

# ─────────────────────────────────────────────
# 4) Build df_for_db and basic QC
# ─────────────────────────────────────────────
df_for_db = df_enrich[db_cols].copy()

print("\nV5-08 — df_for_db shape:", df_for_db.shape)
print("V5-08 — unique roi_dir in df_for_db:", df_for_db["roi_dir"].nunique())

dups = df_for_db[df_for_db["roi_dir"].duplicated(keep=False)]
if not dups.empty:
    print("\nV5-08 — WARNING: duplicate roi_dir in df_for_db; sample:")
    print(
        dups[["roi_dir", "bruker_roi_id", "plate_date", "fish_id"]]
        .sort_values(["roi_dir", "bruker_roi_id"])
        .head(40)
    )
    # hard fail to force fixing upstream rather than silently write bad data
    raise ValueError("V5-08: df_for_db has duplicate roi_dir; fix upstream before export.")

# ─────────────────────────────────────────────
# 5) Write outputs
# ─────────────────────────────────────────────
df_enrich.to_csv(FULL_OUT, index=False)
df_for_db.to_csv(DB_OUT, index=False)

print("\nV5-08 — wrote full enrichment to:", FULL_OUT)
print("V5-08 — wrote DB subset to:     ", DB_OUT)

print("\nV5-08 — DB subset head:")
print(df_for_db.head(10))

V5-08 — FULL_OUT: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_v5.csv
V5-08 — DB_OUT:   /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_for_db_v5.csv
V5-08 — df_enrich shape: (976, 69)

V5-08 — DB column selection:
  kept: ['roi_dir', 'bruker_roi_id', 'plate_date', 'plate_id_filled', 'slot_id_filled', 'roi_index_within_slot', 'dataset_slug', 'experiment_folder', 'fish_id', 'fish_number', 'fish_age_hpf', 'roi_anatomy', 'roi_tiffs', 'date_experiment', 'Date imaged', 'date_mount', 'genotype_base_codes', 'genotype_allele_codes', 'treatment_plasmid_plasmid_base_code', 'treatment_rna_rna_base_code', 'genotype_marker_fluor_codes', 'genotype_marker_tag_codes', 'genotype_marker_localizations', 'genotype_marker_fusion_labels', 'all_unique_organelles', 'all_fluor_organelles', 'link_source']
  missing (not in df_enrich): ['ZF female genotype', 'ZF male genotype', 'treatment_marker_fluor_codes',

In [55]:
# V5-09 — organelle / coverage QC for v5 DB subset

import pandas as pd

if "df_for_db" not in globals():
    raise NameError("V5-09: df_for_db not found; run V5-08 first.")

print("V5-09 — df_for_db shape:", df_for_db.shape)

# ─────────────────────────────────────────────
# 1) Organelles present vs missing
# ─────────────────────────────────────────────
missing_mask = df_for_db["all_unique_organelles"].isna() | (
    df_for_db["all_unique_organelles"].astype(str).str.strip() == ""
)

n_total   = len(df_for_db)
n_missing = int(missing_mask.sum())
n_present = n_total - n_missing

print("\nV5-09 — organelle coverage:")
print("  total ROIs:                  ", n_total)
print("  ROIs with organelles present:", n_present)
print("  ROIs missing organelles:     ", n_missing)

# ─────────────────────────────────────────────
# 2) Breakdown by link_source
# ─────────────────────────────────────────────
print("\nV5-09 — missing organelles by link_source:")
print(df_for_db.loc[missing_mask, "link_source"].value_counts(dropna=False))

# ─────────────────────────────────────────────
# 3) Breakdown by genotype / treatment presence
# ─────────────────────────────────────────────
missing = df_for_db.loc[missing_mask].copy()

missing["has_genotype"] = (
    missing["genotype_base_codes"].notna()
    & (missing["genotype_base_codes"].astype(str).str.strip() != "")
)

rna_col = "treatment_rna_rna_base_code" if "treatment_rna_rna_base_code" in missing.columns else None
pl_col  = "treatment_plasmid_plasmid_base_code" if "treatment_plasmid_plasmid_base_code" in missing.columns else None

if rna_col or pl_col:
    parts = []
    if pl_col:
        parts.append(
            missing[pl_col].notna()
            & (missing[pl_col].astype(str).str.strip() != "")
        )
    if rna_col:
        parts.append(
            missing[rna_col].notna()
            & (missing[rna_col].astype(str).str.strip() != "")
        )
    missing["has_treatment"] = False
    for p in parts:
        missing["has_treatment"] = missing["has_treatment"] | p
else:
    missing["has_treatment"] = False

print("\nV5-09 — missing organelles by genotype presence:")
print(missing["has_genotype"].value_counts(dropna=False))

print("\nV5-09 — missing organelles by treatment (plasmid/RNA) presence:")
print(missing["has_treatment"].value_counts(dropna=False))

# ─────────────────────────────────────────────
# 4) Top datasets among no-org rows (debug hotspots)
# ─────────────────────────────────────────────
print("\nV5-09 — top datasets among no-org rows:")
print(
    missing["dataset_slug"]
    .value_counts()
    .head(20)
)

# ─────────────────────────────────────────────
# 5) Sample of no-org rows for eyeballing
# ─────────────────────────────────────────────
cols_show = [
    "roi_dir",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "roi_anatomy",
    "genotype_base_codes",
]
if rna_col:
    cols_show.append(rna_col)
if pl_col:
    cols_show.append(pl_col)
cols_show += [
    "genotype_marker_fluor_codes",
    "genotype_marker_localizations",
    "treatment_marker_fluor_codes",
    "treatment_marker_localizations",
    "all_unique_organelles",
    "all_fluor_organelles",
    "link_source",
]

print("\nV5-09 — sample missing-org rows:")
print(missing[cols_show].head(30))

V5-09 — df_for_db shape: (976, 27)

V5-09 — organelle coverage:
  total ROIs:                   976
  ROIs with organelles present: 688
  ROIs missing organelles:      288

V5-09 — missing organelles by link_source:
link_source
sheet         170
unmatched     112
date_match      6
Name: count, dtype: int64

V5-09 — missing organelles by genotype presence:
has_genotype
True     253
False     35
Name: count, dtype: int64

V5-09 — missing organelles by treatment (plasmid/RNA) presence:
has_treatment
False    288
Name: count, dtype: int64

V5-09 — top datasets among no-org rows:
dataset_slug
20250513_skittles                            44
20251002_mem_mito                            24
20250917_mem-mito                            21
20250522_skittlez                            19
20250915_mem-mito                            18
20250924_mem-mito                            18
20251107_mem-mito                            13
20251010_mem-mito                            12
20251006_mem_histone 

KeyError: "['treatment_marker_fluor_codes', 'treatment_marker_localizations'] not in index"